
# Smart MCQ Solver Challenge

Models:

1. BiLSTM (from scratch) - 5 epochs
2. DistilBERT - 3 epochs
3. RoBERTa - 3 epochs

Key fixes:

- `eval_strategy="epoch"` for latest Transformers versions
- `fp16=False` for Kaggle GPU compatibility
- `wandb.finish()` is called **after predictions**
- Prediction happens while the WandB run is still active
- Shared training function for DistilBERT and RoBERTa


In [ ]:

import random
import numpy as np
import pandas as pd
import torch
import transformers
import wandb

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

label2id = {'A':0,'B':1,'C':2,'D':3,'E':4}
id2label = {v:k for k,v in label2id.items()}

train["label"] = train["answer"].map(label2id)

train_df, valid_df = train_test_split(
    train,
    test_size=0.2,
    stratify=train["label"],
    random_state=SEED
)


In [ ]:

WANDB_API_KEY = "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"

PROJECT = "23f2000003-t22026"
ENTITY = "23f2000003-t22026"

wandb.login(key=WANDB_API_KEY)


In [ ]:

def map3_score(y_true, preds):

    scores = []

    for y, pred in zip(y_true, preds):

        score = 0

        for rank, p in enumerate(pred[:3], start=1):

            if p == y:
                score = 1 / rank
                break

        scores.append(score)

    return np.mean(scores)


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    top3 = np.argsort(-logits, axis=1)[:, :3]

    return {
        "accuracy": accuracy_score(labels, top3[:, 0]),
        "weighted_f1": f1_score(labels, top3[:, 0], average="weighted"),
        "map3": map3_score(labels, top3)
    }


In [ ]:

def build_dataset(model_name):

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def preprocess(example):

        result = tokenizer(
            [example["prompt"]] * 5,
            [example["A"], example["B"], example["C"], example["D"], example["E"]],
            truncation=True,
            padding="max_length",
            max_length=256
        )

        result["labels"] = example["label"]

        return result

    train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
    valid_ds = Dataset.from_pandas(valid_df.reset_index(drop=True))
    test_ds = Dataset.from_pandas(test.reset_index(drop=True))

    train_ds = train_ds.map(preprocess)
    valid_ds = valid_ds.map(preprocess)

    def preprocess_test(example):

        return tokenizer(
            [example["prompt"]] * 5,
            [example["A"], example["B"], example["C"], example["D"], example["E"]],
            truncation=True,
            padding="max_length",
            max_length=256
        )

    test_ds = test_ds.map(preprocess_test)

    keep_train = ["input_ids", "attention_mask", "labels"]
    keep_test = ["input_ids", "attention_mask"]

    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep_train])
    valid_ds = valid_ds.remove_columns([c for c in valid_ds.column_names if c not in keep_train])
    test_ds = test_ds.remove_columns([c for c in test_ds.column_names if c not in keep_test])

    train_ds.set_format("torch")
    valid_ds.set_format("torch")
    test_ds.set_format("torch")

    return tokenizer, train_ds, valid_ds, test_ds


In [ ]:

def train_transformer(model_name, run_name, epochs):

    wandb.init(
        project=PROJECT,
        entity=ENTITY,
        name=run_name,
        reinit=True
    )

    tokenizer, train_ds, valid_ds, test_ds = build_dataset(model_name)

    model = AutoModelForMultipleChoice.from_pretrained(model_name)

    args = TrainingArguments(
        output_dir=f"./{run_name}",
        run_name=run_name,

        num_train_epochs=epochs,
        learning_rate=2e-5,

        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,

        eval_strategy="epoch",
        save_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="map3",
        greater_is_better=True,

        report_to="wandb",

        fp16=False,
        bf16=False
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    predictions = trainer.predict(test_ds)

    top3 = np.argsort(-predictions.predictions, axis=1)[:, :3]

    submission = pd.DataFrame({
        "id": test["id"],
        "Prediction": [
            " ".join(id2label[x] for x in row)
            for row in top3
        ]
    })

    submission.to_csv(f"{run_name}_submission.csv", index=False)

    wandb.finish()

    return submission


## Model 1 - DistilBERT

In [ ]:

distilbert_submission = train_transformer(
    model_name="distilbert-base-uncased",
    run_name="distilbert",
    epochs=3
)

distilbert_submission.head()


## Model 2 - RoBERTa

In [ ]:

roberta_submission = train_transformer(
    model_name="roberta-base",
    run_name="roberta",
    epochs=3
)

roberta_submission.head()



## Model 3 - BiLSTM



